## Theoretical Foundations of Monte Carlo Methods in Quantitative Finance

In the context of computational finance and probability theory, Monte Carlo (MC) simulation represents a broad class of computational algorithms that rely on repeated random sampling to obtain numerical results. The underlying concept is to use randomness to solve problems that might be deterministic in principle but are analytically intractable, particularly high-dimensional integrals associated with the pricing of complex financial derivatives.

What follows is a formalized theoretical note detailing the mathematical foundations, convergence properties, and application to stochastic differential equations (SDEs) within a risk-neutral pricing framework.

### 1. Mathematical Framework and the Monte Carlo Estimator

The fundamental problem addressed by Monte Carlo integration is the evaluation of a multi-dimensional definite integral, which can be recast as the expectation of a random variable. 

Let $X$ be a random variable defined on a probability space $(\Omega, \mathcal{F}, \mathbb{P})$ with a probability density function $p(x)$. We wish to evaluate the expected value of a measurable function $f(X)$, denoted as $I$:

$$I = \mathbb{E}[f(X)] = \int_{\Omega} f(x)p(x)dx$$

To estimate $I$, the Monte Carlo method draws $N$ independent and identically distributed (i.i.d.) samples, $X_1, X_2, \dots, X_N$, from the distribution $p(x)$. The Monte Carlo estimator, $\hat{I}_N$, is defined as the empirical sample mean:

$$\hat{I}_N = \frac{1}{N} \sum_{i=1}^{N} f(X_i)$$

Since the samples are drawn from the true underlying distribution, $\hat{I}_N$ is an unbiased estimator of $I$, meaning $\mathbb{E}[\hat{I}_N] = I$.

### 2. Convergence and Error Analysis

The theoretical justification for the Monte Carlo method relies on two foundational theorems of probability theory: the Law of Large Numbers (LLN) and the Central Limit Theorem (CLT).

**The Strong Law of Large Numbers (SLLN):**
Provided that $\mathbb{E}[|f(X)|] < \infty$, the SLLN guarantees that the estimator converges almost surely to the true expected value as the number of samples approaches infinity:

$$\mathbb{P}\left( \lim_{N \to \infty} \hat{I}_N = I \right) = 1$$

**The Central Limit Theorem (CLT):**
To quantify the rate of convergence and construct confidence intervals, we apply the CLT. Assuming that the variance of the function is finite, $\sigma^2 = \text{Var}(f(X)) < \infty$, the distribution of the normalized error converges in distribution to a standard normal variable:

$$\sqrt{N} (\hat{I}_N - I) \xrightarrow{d} \mathcal{N}(0, \sigma^2)$$

This implies that the standard error of the Monte Carlo estimator is given by:

$$\text{SE}(\hat{I}_N) = \frac{\sigma}{\sqrt{N}}$$

Crucially, the convergence rate is $\mathcal{O}(N^{-1/2})$, irrespective of the dimension of the state space. This dimension-independence is the primary reason Monte Carlo methods dominate deterministic quadrature rules (e.g., Simpson's rule) in high-dimensional financial problems, such as pricing path-dependent or multi-asset options.

### 3. Application to Asset Pricing (Risk-Neutral Framework)

In quantitative finance, the Fundamental Theorem of Asset Pricing states that the arbitrage-free price of a derivative is the discounted expected value of its future payoff under the risk-neutral measure $\mathbb{Q}$.

Consider an underlying asset whose price dynamics $S_t$ follow a Geometric Brownian Motion (GBM) under $\mathbb{Q}$:

$$dS_t = r S_t dt + \sigma S_t dW_t^{\mathbb{Q}}$$

Where:
* $r$ is the risk-free interest rate.
* $\sigma$ is the constant volatility.
* $W_t^{\mathbb{Q}}$ is a standard Wiener process under $\mathbb{Q}$.

By applying Itô's Lemma to $\ln(S_t)$, the SDE can be integrated exactly over a time step $\Delta t = T - t$, yielding the exact discretization:

$$S_T = S_t \exp\left( \left(r - \frac{1}{2}\sigma^2\right)\Delta t + \sigma \sqrt{\Delta t} Z \right)$$

Where $Z \sim \mathcal{N}(0,1)$ is a standard normal random variable. 

To price a European option with a payoff function $\Phi(S_T)$ maturing at $T$, we simulate $N$ terminal asset prices $\{S_{T}^{(i)}\}_{i=1}^N$ using $N$ independent draws from $\mathcal{N}(0,1)$. The Monte Carlo price $V_0$ is the discounted average payoff:

$$V_0 \approx e^{-rT} \frac{1}{N} \sum_{i=1}^{N} \Phi(S_{T}^{(i)})$$

### 4. Estimating π via Monte Carlo Simulation

To estimate π, generate N independent uniformly distributed points (X_i, Y_i) over the square [-r, r] × [-r, r] (commonly with r = 1). The probability that a random point falls inside the inscribed circle is π/4. Define the indicator I(X_i^2 + Y_i^2 ≤ r^2). The Monte Carlo estimator for π is

$$\hat{\pi}_N = 4 \cdot \frac{1}{N} \sum_{i=1}^N \mathbb{I}(X_i^2 + Y_i^2 \le r^2).$$

For r = 1 this simplifies to 4 times the sample mean of the indicators. By the Strong Law of Large Numbers, \hat{\pi}_N → π almost surely as N → ∞.

### 5. Variance Reduction (Overview)

Because computational cost scales quadratically with required precision (due to the $\sqrt{N}$ convergence), variance reduction techniques are standard practice to reduce $\sigma^2$ without increasing $N$. Prominent methods include:

* **Antithetic Variates:** Utilizing negatively correlated sample pairs (e.g., simulating paths with both $Z$ and $-Z$).
* **Control Variates:** Exploiting the known analytical expected value of a highly correlated, simpler derivative to correct the simulation error of a complex one.
* **Importance Sampling:** Changing the probability measure (via the Radon-Nikodym derivative) to sample more frequently in regions that contribute most significantly to the expected value, heavily utilized in deep out-of-the-money option pricing and Value-at-Risk (VaR) calculations.

In [2]:
%pip install ipympl
%matplotlib tk
import numpy as np
import matplotlib.pyplot as plt

r = 1.0
N = 100000
step = 200

inside = 0
xs_inside, ys_inside = [], []
xs_out, ys_out = [], []
counts, estimates = [], []

# 1. Attiva la modalità interattiva di Matplotlib
plt.ion()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

# --- Configurazione Iniziale Ax1 (Il Cerchio) ---
theta = np.linspace(0, 2 * np.pi, 200)
ax1.plot(r * np.cos(theta), r * np.sin(theta), color="black")
# Creiamo oggetti scatter vuoti che riempiremo dopo
scat_inside = ax1.scatter([], [], s=4, color="blue", alpha=0.6, label="inside")
scat_out = ax1.scatter([], [], s=4, color="red", alpha=0.6, label="outside")
ax1.axis("equal")
ax1.set_xlim(-r, r)
ax1.set_ylim(-r, r)
title_ax1 = ax1.set_title(f"Monte Carlo Circle Area\nN=0, estimate=0.0")
ax1.legend(loc="upper right", fontsize=8)

# --- Configurazione Iniziale Ax2 (La Convergenza) ---
line_est, = ax2.plot([], [], "-o", markersize=3)
ax2.axhline(np.pi, color="gray", linestyle="--", label="π")
ax2.set_xlim(0, N)
ax2.set_ylim(2.5, 4.0) # Fissiamo i limiti della Y per evitare salti visivi
ax2.set_xlabel("Number of samples")
ax2.set_ylabel("Area estimate")
ax2.set_title("Convergence of Estimate")
ax2.grid(True)
ax2.legend()

plt.show()

# 2. Ciclo di simulazione
for i in range(1, N + 1):
    x, y = np.random.uniform(-r, r, 2)
    
    if x**2 + y**2 <= r**2:
        inside += 1
        xs_inside.append(x)
        ys_inside.append(y)
    else:
        xs_out.append(x)
        ys_out.append(y)

    # 3. Aggiornamento Live del grafico
    if i % step == 0 or i == N:
        estimate = 4 * inside / i
        counts.append(i)
        estimates.append(estimate)

        # Aggiorniamo solo i dati (molto più veloce di plt.clf!)
        if xs_inside:
            scat_inside.set_offsets(np.c_[xs_inside, ys_inside])
        if xs_out:
            scat_out.set_offsets(np.c_[xs_out, ys_out])
            
        title_ax1.set_text(f"Monte Carlo Circle Area\nN={i}, estimate={estimate:.6f}")
        line_est.set_data(counts, estimates)

        # Disegna l'aggiornamento sullo schermo
        fig.canvas.draw()
        fig.canvas.flush_events()

# Disattiva la modalità interattiva e mantieni aperto il grafico finale
plt.ioff()
print("Final area estimate:", estimate)
plt.show()

Note: you may need to restart the kernel to use updated packages.


Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.


Final area estimate: 3.14324
